# 📈 Sistema de Trading Automatizado v9.1 – GEBRA Portfolio
**Novo:** Wyckoff completo (ligado/desligado), log detalhado, Telegram, e-mail

In [ ]:
import os
EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')

# ==================== RECURSO WYCKOFF COMPLETO ====================
USAR_WYCKOFF_COMPLETO = True    # True = metodologia oficial, False = desligado
WYCKOFF_SENSIBILIDADE = 80      # 5 a 200 (5=muito sensível, 200=muito restritivo)
# ==================================================================

PARAMS_BAIXA_VOL = {
    'kelly_frac': 0.30, 'wyckoff_threshold': 0.75, 'gap_max_pct': 0.055,
    'custos_pct': 0.003, 'exigir_volume_anormal': False,
    'risco_percentual_maximo': 0.15, 'preco_minimo': 2.00
}
PARAMS_ALTA_VOL = {
    'kelly_frac': 0.15, 'wyckoff_threshold': 0.85, 'gap_max_pct': 0.03,
    'custos_pct': 0.006, 'exigir_volume_anormal': True,
    'risco_percentual_maximo': 0.10, 'preco_minimo': 2.00
}
PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()
MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30
HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v85.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v85.log"
CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA', 'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']
FALLBACK_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3', 'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3', 'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3']
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 5.00
BANDA_ZONA_PCT = 0.01
EXIGIR_CONFLUENCIA_CANDLE = True
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
CACHE_MACRO_EXPIRY_HORAS = 24
ALTA_CONFIABILIDADE = False
DIST_CORDA_MAX = 30.0
USAR_GATILHO_BOLLINGER = False
USAR_GUARDIAO_MACD = False
MAX_ATIVOS_POR_SETOR = 2
MODO_GEBRA = 'essencial'
LOG_DETALHADO_TICKER = True
LOG_PERFORMANCE = True
LOG_FILTROS_DETALHADO = True
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
EXIGIR_CONFLUENCIA = True

print("✅ Parâmetros v9.1 carregados")
print(f"   Wyckoff Completo: {'LIGADO' if USAR_WYCKOFF_COMPLETO else 'DESLIGADO'} | Sensibilidade: {WYCKOFF_SENSIBILIDADE}")

In [ ]:
!pip install yfinance pandas-ta python-dotenv requests-cache requests-ratelimiter gspread oauth2client --quiet 2>/dev/null
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os, sys, traceback, gc, subprocess
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter
from requests_cache import CachedSession
from requests_ratelimiter import LimiterSession
import gspread
from oauth2client.service_account import ServiceAccountCredentials
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from dotenv import load_dotenv
    load_dotenv()
    if not EMAIL_REMETENTE: EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
    if not SENHA_APP: SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
except ImportError: pass
try:
    session = LimiterSession(per_second=0.4, session_factory=lambda: CachedSession(cache_name='yfinance.cache', backend='sqlite', expire_after=timedelta(hours=6)))
    yf.shared._requests = session
    print("✅ Cache YFinance ativado")
except Exception as e: print(f"⚠️ Cache: {e}")

class Logger:
    def __init__(self, al, ad=None):
        self.al = al; self.ad = ad; self.t0 = time.time()
        self.tm = {}; self.ct = {}; self.bf = []; self.mb = 100
    def log(self, m, n="INFO", t=None):
        ts = datetime.now().strftime("%H:%M:%S")
        msg = f"[{ts}] [{n}] {m}"
        if t: msg += f" | {t}"
        print(msg)
        if self.ad and LOG_PERFORMANCE:
            self.bf.append(msg+"\n")
            if len(self.bf) >= self.mb: self._fb()
    def _fb(self):
        if self.ad and self.bf:
            try:
                with open(self.ad,'a',encoding='utf-8') as f: f.writelines(self.bf)
                self.bf.clear()
            except Exception as e: print(f"Erro log: {e}")
    def warn(self,m,t=None): self.log(m,"WARN",t)
    def error(self,m,t=None): self.log(m,"ERRO",t)
    def inicio(self,n): self.tm[n]={'ini':time.time()}; self.log(f"🚀 {n}","ETAPA")
    def fim(self,n,d=None):
        if n in self.tm:
            dr = time.time()-self.tm[n]['ini']; self.tm[n]['dur']=dr
            msg = f"✅ {n} ({dr:.1f}s)"
            if d: msg += " | "+" | ".join(f"{k}:{v}" for k,v in d.items())
            self.log(msg,"ETAPA")
    def resumo(self):
        self._fb(); tt = time.time()-self.t0
        self.log("\n"+"="*60,"RESUMO"); self.log(f"⏱️ Total: {tt:.1f}s","RESUMO")
        for e,d in self.tm.items():
            if 'dur' in d: self.log(f"   • {e}: {d['dur']:.1f}s ({d['dur']/tt*100:.0f}%)","RESUMO")
        self.log("="*60+"\n","RESUMO")

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def _log_exc(c, e):
    try:
        msg = f"[{c}] {type(e).__name__}: {str(e)[:200]}"
        if 'logger' in globals(): logger.error(msg)
        else: print(f"[ERRO] {msg}")
        with open('traceback_errors.log','a',encoding='utf-8') as f:
            f.write(f"\n{'='*60}\n{datetime.now()}\n{c}\n{str(e)}\n{traceback.format_exc()}")
    except: pass

# Google Sheets
def conectar_sheets():
    try:
        from google.colab import userdata
        kc = userdata.get('GCP_SERVICE_ACCOUNT_KEY')
        if not kc: return None,None,None
        jk = json.loads(kc)
        sc = ['https://spreadsheets.google.com/feeds','https://www.googleapis.com/auth/drive']
        cr = ServiceAccountCredentials.from_json_keyfile_dict(jk,sc)
        cl = gspread.authorize(cr)
        pid = userdata.get('GOOGLE_SHEET_ID')
        pl = cl.open_by_key(pid)
        return pl.worksheet("CircuitBreaker"),pl.worksheet("Ledger"),pl.worksheet("Scanner")
    except Exception as e:
        logger.warn(f"Sheets: {e}")
        return None,None,None

aba_circuit, aba_ledger, aba_scanner = conectar_sheets()

# Telegram
try:
    from google.colab import userdata
    TELEGRAM_TOKEN = userdata.get('TELEGRAM_TOKEN')
    TELEGRAM_CHAT_ID = userdata.get('TELEGRAM_CHAT_ID')
except:
    TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN','')
    TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID','')

def enviar_telegram(m, pm='HTML'):
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID: return
    try: requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage", data={'chat_id':TELEGRAM_CHAT_ID,'text':m,'parse_mode':pm}, timeout=10)
    except Exception as e: _log_exc('Telegram',e)

print("✅ Célula 1 carregada")

In [ ]:
# CÉLULA 2: FUNÇÕES AUXILIARES (Mantidas da v9.0)
def _safe_divide(a,b,d=np.nan):
    if b is None or b==0 or pd.isna(b): return d
    return a/b
def _safe_log(x,d=np.nan):
    if x is None or x<=0 or pd.isna(x): return d
    return np.log(x)
def calcular_eficiencia_candle(df):
    corpo = abs(df['Close']-df['Open'])
    ss = df['High']-df[['Close','Open']].max(axis=1)
    si = df[['Close','Open']].min(axis=1)-df['Low']
    rt = df['High']-df['Low']
    rt = rt.replace(0,np.nan)
    ef = pd.Series(index=df.index,dtype=float)
    al = df['Close']>df['Open']
    bx = df['Close']<df['Open']
    ef[al] = 1-(ss[al]/rt[al])
    ef[bx] = 1-(si[bx]/rt[bx])
    return ef
def detectar_regime(df,j=20):
    dt = df.copy()
    dt['ret'] = dt['Close'].pct_change()
    dt['vol'] = dt['ret'].rolling(j).std()
    try:
        ax = ta.adx(dt['High'],dt['Low'],dt['Close'],length=14)
        if ax is not None and not ax.empty:
            if isinstance(ax,pd.DataFrame) and 'ADX_14' in ax.columns: dt['adx']=ax['ADX_14']
            else: dt['adx']=ax.iloc[:,0] if len(ax.shape)>1 else ax
        else: return pd.Series(index=df.index,dtype=int)
    except Exception as e: _log_exc('regime',e); return pd.Series(index=df.index,dtype=int)
    dt = dt.dropna(subset=['vol','adx'])
    if dt.empty: return pd.Series(index=df.index,dtype=int)
    vp33,vp67 = dt['vol'].quantile([0.33,0.67])
    ap33,ap67 = dt['adx'].quantile([0.33,0.67])
    def cl(row):
        v,a = row['vol'],row['adx']
        if v<vp33 and a<ap33: return 0
        elif v>vp67 or a>ap67: return 2
        return 1
    rg = dt.apply(cl,axis=1)
    rs = pd.Series(index=df.index,dtype=int)
    rs.loc[rg.index] = rg
    rs.ffill(inplace=True)
    return rs
def detectar_swing_low(df,j=10,conf=True):
    if len(df)<j: return float(df['Low'].min())
    lo = df['Low']
    mr = lo.rolling(window=2*j+1,center=True).min()
    sw = (lo==mr)&~mr.isna()
    if conf: sw = sw&(df['Close'].shift(-1)>lo)
    sv = lo[sw]
    return float(sv.iloc[-1]) if len(sv)>0 else float(lo.min())
def detectar_swing_high(df,j=10,conf=True):
    if len(df)<j: return float(df['High'].max())
    hi = df['High']
    mr = hi.rolling(window=2*j+1,center=True).max()
    sw = (hi==mr)&~mr.isna()
    if conf: sw = sw&(df['Close'].shift(-1)<hi)
    sv = hi[sw]
    return float(sv.iloc[-1]) if len(sv)>0 else float(hi.max())
def calcular_lta_pivos(df,jp=5):
    lo = df['Low']
    if len(lo)<2*jp+1: return None
    ll = np.log(lo.replace(0,np.nan))
    mr = ll.rolling(window=2*jp+1,center=True).min()
    pm = (ll==mr)&~mr.isna()
    ip = np.where(pm)[0]
    if len(ip)<2: return None
    x1,x2 = ip[-2],ip[-1]
    y1,y2 = ll.iloc[x1],ll.iloc[x2]
    if x2==x1: return None
    return np.exp(y2+(y2-y1)/(x2-x1)*(len(ll)-1-x2))
def calcular_ltb_pivos(df,jp=5):
    hi = df['High']
    if len(hi)<2*jp+1: return None
    lh = np.log(hi.replace(0,np.nan))
    mr = lh.rolling(window=2*jp+1,center=True).max()
    pm = (lh==mr)&~mr.isna()
    ip = np.where(pm)[0]
    if len(ip)<2: return None
    x1,x2 = ip[-2],ip[-1]
    y1,y2 = lh.iloc[x1],lh.iloc[x2]
    if x2==x1 or y2>=y1: return None
    return np.exp(y2+(y2-y1)/(x2-x1)*(len(lh)-1-x2))
def detectar_armadilha_lta(df,lta,bp=0.01):
    if len(df)<2 or lta is None or lta<=0: return False,0.0
    ul = df.iloc[-1]; lo,cl,op = ul['Low'],ul['Close'],ul['Open']
    co = abs(cl-op); rc = ul['High']-lo
    if rc<=0: return False,0.0
    si = min(cl,op)-lo; zi = lta*(1-bp)
    if lo<zi and cl>=zi and si>=2*co: return True,min(1.0,si/rc)
    return False,0.0
def detectar_armadilha_ltb(df,ltb,bp=0.01):
    if len(df)<2 or ltb is None or ltb<=0: return False,0.0
    ul = df.iloc[-1]; hi,cl,op = ul['High'],ul['Close'],ul['Open']
    co = abs(cl-op); rc = hi-ul['Low']
    if rc<=0: return False,0.0
    ss = hi-max(cl,op); zs = ltb*(1+bp)
    if hi>zs and cl<=zs and ss>=2*co: return True,min(1.0,ss/rc)
    return False,0.0
def calcular_lta_adaptativo(df,js=None):
    if js is None: js=[20,30,40,50]
    if len(df)<max(js): return None
    lo,cl = df['Low'].values,df['Close'].values
    ms,mb = -np.inf,None
    for j in js:
        lta = calcular_lta_pivos(df,jp=j)
        if lta is None: continue
        zi = lta*(1-BANDA_ZONA_PCT); zs = lta*(1+BANDA_ZONA_PCT)
        ini = max(0,len(lo)-max(2*j,50))
        sc = (np.sum(lo[ini:]>=zi)-3*np.sum(cl[ini:]<zi)+2*np.sum((lo[ini:]>=zi)&(lo[ini:]<=zs)))
        if sc>ms: ms,mb = sc,(lta,j)
    return mb
def calcular_ltb_adaptativo(df,js=None):
    if js is None: js=[20,30,40,50]
    if len(df)<max(js): return None
    hi,cl = df['High'].values,df['Close'].values
    ms,mb = -np.inf,None
    for j in js:
        ltb = calcular_ltb_pivos(df,jp=j)
        if ltb is None: continue
        zi = ltb*(1-BANDA_ZONA_PCT); zs = ltb*(1+BANDA_ZONA_PCT)
        ini = max(0,len(hi)-max(2*j,50))
        sc = (np.sum(hi[ini:]<=zs)-3*np.sum(cl[ini:]>zs)+2*np.sum((hi[ini:]>=zi)&(hi[ini:]<=zs)))
        if sc>ms: ms,mb = sc,(ltb,j)
    return mb
def validar_elliott(df,sl,sh):
    if len(sl)<5 or len(sh)<5: return False,"Pivôs"
    try:
        o1l,o1h = sl[-5],sh[-4]; o2l,o3h = sl[-4],sh[-3]
        o4l,o5h = sl[-3],sh[-2]
        if o2l<o1l: return False,"Onda2"
        if o4l<o1h: return False,"Onda4"
        a1,a3,a5 = abs(o1h-o1l),abs(o3h-o2l),abs(o5h-o4l)
        if a3<=min(a1,a5): return False,"Onda3"
        return True,"1-2-3-4-5"
    except Exception as e: _log_exc('elliott',e); return False,"Erro"
def calcular_obv_divergencia(df):
    if len(df)<20: return None
    try:
        obv = ta.obv(df['Close'],df['Volume'])
        if obv is None or len(obv)<20: return None
        pr = df['Close'].iloc[-20:].values; ov = obv.iloc[-20:].values
        if pr[-1]<pr[0] and ov[-1]>ov[0]: return 'alta'
        if pr[-1]>pr[0] and ov[-1]<ov[0]: return 'baixa'
        return None
    except Exception as e: _log_exc('obv',e); return None
def calcular_willr(df,p=14):
    try:
        w = ta.willr(df['High'],df['Low'],df['Close'],length=p)
        if w is not None and not w.empty:
            v = w.iloc[-1]; return round(float(v),1) if pd.notna(v) else None
    except Exception as e: _log_exc('willr',e)
    return None
def calcular_rsi(df,p=14):
    try:
        r = ta.rsi(df['Close'],length=p)
        if r is not None and not r.empty:
            v = r.iloc[-1]; return round(float(v),1) if pd.notna(v) else None
    except Exception as e: _log_exc('rsi',e)
    return None
def calcular_macd(df):
    try:
        m = ta.macd(df['Close'])
        if m is not None and not m.empty:
            mv,sv,hv = m['MACD_12_26_9'].iloc[-1],m['MACDs_12_26_9'].iloc[-1],m['MACDh_12_26_9'].iloc[-1]
            return (round(float(mv),2) if pd.notna(mv) else None,round(float(sv),2) if pd.notna(sv) else None,round(float(hv),2) if pd.notna(hv) else None)
    except Exception as e: _log_exc('macd',e)
    return None,None,None
def calcular_estocastico(df,p=14):
    try:
        s = ta.stoch(df['High'],df['Low'],df['Close'],k=p,d=3)
        if s is not None and not s.empty:
            k,d = s['STOCHk_14_3_3'].iloc[-1],s['STOCHd_14_3_3'].iloc[-1]
            return (round(float(k),1) if pd.notna(k) else None,round(float(d),1) if pd.notna(d) else None)
    except Exception as e: _log_exc('estoc',e)
    return None,None
def calcular_bandas_bollinger(df,p=20):
    try:
        bb = ta.bbands(df['Close'],length=p)
        if bb is not None and not bb.empty:
            u,m,l = float(bb['BBU_20_2.0'].iloc[-1]),float(bb['BBM_20_2.0'].iloc[-1]),float(bb['BBL_20_2.0'].iloc[-1])
            c = float(df['Close'].iloc[-1])
            pos = (c-l)/(u-l)*100 if u!=l else 50
            return {'upper':round(u,2),'mid':round(m,2),'lower':round(l,2),'posicao_%':round(pos,1)}
    except Exception as e: _log_exc('bb',e)
    return None
def calcular_climax_volume(df,p=50):
    if len(df)<p: return False
    try:
        vm = df['Volume'].rolling(p).mean().iloc[-1]
        va = df['Volume'].iloc[-1]
        return va>3*vm if (pd.notna(vm) and vm>0) else False
    except Exception as e: _log_exc('climax',e); return False
def analisar_candle(row,ant=None):
    o,h,l,c = row['Open'],row['High'],row['Low'],row['Close']
    co = abs(c-o); rt = h-l
    if rt<=0: return {}
    ps,pi = h-max(o,c),min(o,c)-l
    res = {}
    if pi>=2*co and ps<=0.3*rt and co>0: res['martelo']=True
    if ps>=2*co and pi<=0.3*rt and co>0: res['estrela_cadente']=True
    if co<=0.05*rt: res['doji']=True
    if ant is not None:
        oa,ca = ant['Open'],ant['Close']; ca_ant = abs(ca-oa)
        if co<ca_ant and h<=ant['High'] and l>=ant['Low']:
            if ca<oa and c>o: res['harami_alta']=True
            elif ca>oa and c<o: res['harami_baixa']=True
        if co>ca_ant:
            if ca<oa and c>o and o<=ca and c>=oa: res['engolfo_alta']=True
            if ca>oa and c<o and o>=ca and c<=oa: res['engolfo_baixa']=True
        if ca<oa and c>o and o>ca: res['kicker_alta']=True
        if ca>oa and c<o and o<ca: res['kicker_baixa']=True
    return res
def detectar_bebe_abandonado(df):
    if len(df)<4: return None
    c1,c2,c3 = df.iloc[-4],df.iloc[-3],df.iloc[-2]
    if c1['High']<=0 or c2['High']<=0 or c2['Low']<=0 or c3['Low']<=0: return None
    try:
        if c1['Close']<c1['Open']: g1 = (c2['Low']-c1['High'])/c1['High']
        else: g1 = (c1['Low']-c2['High'])/c2['High']
        if c3['Close']>c3['Open']: g2 = (c3['High']-c2['Low'])/c2['Low']
        else: g2 = (c2['High']-c3['Low'])/c3['Low']
    except: return None
    if g1>0.02 and g2>0.02 and analisar_candle(c2).get('doji'):
        if c1['Close']<c1['Open'] and c3['Close']>c3['Open']: return {'tipo':'Bebe_abandonado_alta'}
        if c1['Close']>c1['Open'] and c3['Close']<c3['Open']: return {'tipo':'Bebe_abandonado_baixa'}
    return None
def detectar_retangulo(df,j=20):
    if len(df)<j: return None
    hh,ll = df['High'].iloc[-j:],df['Low'].iloc[-j:]
    r,s = hh.max(),ll.min()
    if r-s<0.02*s: return None
    if np.sum(hh.values>=r*0.99)>=2 and np.sum(ll.values<=s*1.01)>=2:
        return {'tipo':'Retangulo','suporte':round(s,2),'resistencia':round(r,2)}
    return None
def detectar_alargamento(df,j=20):
    if len(df)<j: return None
    hh,ll = df['High'].iloc[-j:].values,df['Low'].iloc[-j:].values
    if hh[-1]>hh[0] and ll[-1]<ll[0]: return {'tipo':'Alargamento'}
    return None
def detectar_estrutura_dow(df):
    if len(df)<26: return None
    ut,pt = df['High'].iloc[-1],df['High'].iloc[-26]
    uf,pf = df['Low'].iloc[-1],df['Low'].iloc[-26]
    if ut>pt and uf>pf: return {'tendencia_dow':'ALTA'}
    elif ut<pt and uf<pf: return {'tendencia_dow':'BAIXA'}
    return {'tendencia_dow':'LATERAL'}
def calcular_fibonacci_retracao(df):
    if len(df)<50: return None
    sh = detectar_swing_high(df,10,False); sl = detectar_swing_low(df,10,False)
    if sh is None or sl is None or sh<=sl or sl<=0: return None
    diff = sh-sl
    return {'38.2%':round(sh-diff*0.382,2),'50.0%':round(sh-diff*0.5,2),'61.8%':round(sh-diff*0.618,2)}
def calcular_alvos_fibonacci(df,dir,fw=20):
    if len(df)<fw: return {}
    try:
        dfr = df.iloc[-fw:]
        sl = detectar_swing_low(dfr,5,False); sh = detectar_swing_high(dfr,5,False)
        if sl is None or sh is None or sl<=0 or sh<=0 or sl>=sh: return {}
        amp = np.log(sh)-np.log(sl); base = np.log(sh); mults = [1.0,1.618,2.618,4.236]
        return {f'{m*100}%':round(np.exp(base+amp*m),2) for m in mults} if dir=='COMPRA' else {f'{m*100}%':round(np.exp(base-amp*m),2) for m in mults}
    except Exception as e: _log_exc('fib',e); return {}
def calcular_payoff_real(ent,alv,stp,cst,dir='COMPRA'):
    if dir=='COMPRA':
        if stp>=ent or alv<=ent: return 0.0
        risco = ent-stp; ret = alv-ent
    elif dir=='VENDA':
        if stp<=ent or alv>=ent: return 0.0
        risco = stp-ent; ret = ent-alv
    else: return 0.0
    if risco<=0: return 0.0
    return round(max(0,ret-(ent+alv)*cst)/risco,2)
def detectar_regime_volatilidade(serie,j=40):
    try:
        if serie is None or len(serie)<j: return 'BAIXA'
        ret = serie.pct_change().dropna()
        if len(ret)<20: return 'BAIXA'
        vol_at = float(ret.rolling(20).std().iloc[-1])
        vol_hist = ret.rolling(j).std().dropna()
        if vol_hist.empty: return 'BAIXA'
        return 'ALTA' if float((vol_hist<vol_at).mean())>0.7 else 'BAIXA'
    except Exception as e: _log_exc('volat',e); return 'BAIXA'
def detectar_volume_anormal(df,p=20,lim=1.5):
    if len(df)<p: return False
    try:
        vm = df['Volume'].rolling(p).mean().iloc[-1]
        va = df['Volume'].iloc[-1]
        return va>=vm*lim if (pd.notna(vm) and vm>0) else False
    except Exception as e: _log_exc('vol',e); return False
def fractional_kelly(wr,pr,frac=0.25):
    if pr<=0: return 0.0
    kelly = (pr*wr-(1-wr))/pr
    return max(0.0,min(kelly,0.25))*frac
print("✅ Célula 2 carregada")

In [ ]:
# ==================== CÉLULA 3: GUARDIÕES + WYCKOFF COMPLETO ====================
SETOR_POR_TICKER = {
    'PETR4':'Petróleo','PETR3':'Petróleo','PRIO3':'Petróleo',
    'VALE3':'Mineração','GGBR4':'Siderurgia','CSNA3':'Siderurgia',
    'ITUB4':'Financeiro','BBDC4':'Financeiro','BBAS3':'Financeiro',
    'ABEV3':'Consumo','MGLU3':'Varejo','RENT3':'Varejo',
    'WEGE3':'Indústria','RADL3':'Saúde','JBSS3':'Alimentos',
}
def obter_setor(t): return SETOR_POR_TICKER.get(t.replace('.SA',''),'Outros')

MACRO_REFERENCE = {'VALE3.SA':('GC=F',0.6),'PETR4.SA':('CL=F',0.8),'PETR3.SA':('CL=F',0.8),'ABEV3.SA':('CORN',0.3)}
_cmd,_cmt = {},{}
def _obter_dados_macro(bench):
    agora = datetime.now()
    if bench in _cmt and (agora-_cmt[bench]).total_seconds()/3600<CACHE_MACRO_EXPIRY_HORAS: return _cmd.get(bench)
    try:
        dfb = yf.download(bench,period='1y',interval='1wk',progress=False,auto_adjust=True)
        vals = dfb['Close'].values if not dfb.empty else None
        _cmd[bench],_cmt[bench] = vals,agora
        return vals
    except Exception as e: _log_exc('macro',e); return None
def verificar_alinhamento_macro(t,d,_=None):
    if t not in MACRO_REFERENCE: return True,15
    ref,_ = MACRO_REFERENCE[t]
    pr = _obter_dados_macro(ref)
    if pr is None or len(pr)<150: return True,15
    ema = pd.Series(pr).ewm(span=50,adjust=False).mean()
    ea,el = ema.iloc[-1],ema.iloc[-5]
    if pd.isna(ea) or pd.isna(el): return True,15
    sp = ea>el
    return (True,30) if (d=='COMPRA' and pr[-1]>ea and sp) or (d=='VENDA' and pr[-1]<ea and not sp) else (False,0)

def validar_toque_zona_wyckoff(df,lta,bp=0.01,vmr=0.8):
    if len(df)<2 or lta is None or lta<=0: return False,'indefinido'
    ul = df.iloc[-1]; lo,cl = ul['Low'],ul['Close']; va = ul['Volume']
    vm = df['Volume'].rolling(20).mean().iloc[-1] if len(df)>=20 else va
    zi,zs = lta*(1-bp),lta*(1+bp)
    if not (lo<=zs and cl>=zi): return False,'indefinido'
    sr = df['Low'].rolling(20).min().iloc[-1]; rr = df['High'].rolling(20).max().iloc[-1]
    rt = rr-sr; pos = (cl-sr)/rt if rt>0 else 0.5
    if pos<0.4:
        reg = 'acumulacao'; tv = va>=vm*vmr if (pd.notna(vm) and vm>0) else True
    elif pos>0.6:
        reg = 'markup'; tv = va>=vm*1.2 if (pd.notna(vm) and vm>0) else True
    else:
        reg = 'neutro'; tv = va>=vm*0.8 if (pd.notna(vm) and vm>0) else True
    return tv,reg

# ==================== NOVO: WYCKOFF COMPLETO ====================
def detectar_wyckoff_completo(df_w):
    """Metodologia Wyckoff oficial: Spring, UTAD, fases A-E, esforço vs resultado"""
    if len(df_w)<50: return 'NEUTRO',None,0.0,'Dados insuficientes (<50 velas)'
    cl = df_w['Close'].values; hi = df_w['High'].values
    lo = df_w['Low'].values; vol = df_w['Volume'].values
    jr = min(200,len(df_w))
    rh = np.max(hi[-jr:]); rl = np.min(lo[-jr:])
    pa = cl[-1]; amp = (rh-rl)/rl if rl>0 else 0
    if amp<0.10: return 'NEUTRO',None,0.0,'Range muito estreito'
    pos = (pa-rl)/(rh-rl) if rh>rl else 0.5
    mm200 = np.mean(cl[-min(200,len(df_w)):])
    vm20 = np.mean(vol[-20:]); va = vol[-1]; vr = va/vm20 if vm20>0 else 1.0
    c6m = np.mean(cl[-min(130,len(df_w)):-min(104,len(df_w))]) if len(df_w)>=130 else cl[0]
    ta_ant = 'ALTA' if pa>c6m*1.05 else ('BAIXA' if pa<c6m*0.95 else 'LATERAL')
    spring,sp_f = False,0.0; utad,ut_f = False,0.0
    for i in range(len(df_w)-10,len(df_w)-1):
        if lo[i]<rl*0.98 and cl[i]>rl:
            spring = True
            pen = (rl-lo[i])/rl; rec = (cl[i]-lo[i])/(hi[i]-lo[i]) if hi[i]>lo[i] else 0
            sp_f = min(1.0,(pen+rec)/2)
            break
        if hi[i]>rh*1.02 and cl[i]<rh:
            utad = True
            pen = (hi[i]-rh)/rh; rej = (hi[i]-cl[i])/(hi[i]-lo[i]) if hi[i]>lo[i] else 0
            ut_f = min(1.0,(pen+rej)/2)
            break
    dir,fase,conf,evt = 'NEUTRO',None,0.0,''
    if spring and sp_f>0.3:
        dir = 'COMPRA'; fase = 'C'
        conf = sp_f*(0.5+0.5*(WYCKOFF_SENSIBILIDADE/200))
        evt = f'Spring (força {sp_f:.0%})'
        if va>vm20*1.5: conf+=0.15; evt+=' + clímax de venda'
        if pos<0.3: conf+=0.10; evt+=' | fundo do range'
    elif utad and ut_f>0.3:
        dir = 'VENDA'; fase = 'C'
        conf = ut_f*(0.5+0.5*(WYCKOFF_SENSIBILIDADE/200))
        evt = f'UTAD (força {ut_f:.0%})'
        if va>vm20*1.5: conf+=0.15; evt+=' + clímax de compra'
        if pos>0.7: conf+=0.10; evt+=' | topo do range'
    elif pos<0.25 and ta_ant=='BAIXA':
        dir = 'COMPRA'; fase = 'A/B'
        conf = 0.25*(WYCKOFF_SENSIBILIDADE/200)
        evt = 'Possível Acumulação (fundo do range)'
        vm50 = np.mean(vol[-50:]) if len(df_w)>=50 else vm20
        vm100 = np.mean(vol[-min(100,len(df_w)):-50]) if len(df_w)>=100 else vm20
        if vm50<vm100*0.8: conf+=0.15; evt+=' | volume diminuindo'
    elif pos>0.75 and ta_ant=='ALTA':
        dir = 'VENDA'; fase = 'A/B'
        conf = 0.25*(WYCKOFF_SENSIBILIDADE/200)
        evt = 'Possível Distribuição (topo do range)'
    elif pa>mm200 and pos>0.5:
        dir = 'COMPRA'; fase = 'D/E'
        conf = 0.20*(WYCKOFF_SENSIBILIDADE/200)
        evt = 'Acima MM200 (markup)'
    elif pa<mm200 and pos<0.5:
        dir = 'VENDA'; fase = 'D/E'
        conf = 0.20*(WYCKOFF_SENSIBILIDADE/200)
        evt = 'Abaixo MM200 (markdown)'
    conf = min(1.0,conf*(WYCKOFF_SENSIBILIDADE/100))
    return dir,fase,conf,evt

def guardiao_wyckoff(df_w,lta=None,bp=0.01):
    if USAR_WYCKOFF_COMPLETO:
        dir,fase,conf,evt = detectar_wyckoff_completo(df_w)
        ok = (dir in ['COMPRA','VENDA']) and conf>=0.10
        reg = 'acumulacao' if dir=='COMPRA' else ('distribuicao' if dir=='VENDA' else 'indefinido')
        info = {'fase':fase,'confianca':conf,'evento':evt,'direcao':dir}
        return ok,'Wyckoff',reg,info
    else:
        if lta is None or lta<=0: return True,None,'indefinido',{}
        toca,reg = validar_toque_zona_wyckoff(df_w,lta,bp)
        return toca,'Zona Wyckoff' if not toca else None,reg,{}

def _normalizar_dataframe(df):
    df = df.copy()
    if isinstance(df.columns,pd.MultiIndex): df.columns = ['_'.join(col).strip() for col in df.columns.values]
    rm = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
    for l,o in {str(c).lower():c for c in df.columns}.items():
        if l in rm: df.rename(columns={o:rm[l]},inplace=True)
    df.sort_index(inplace=True)
    if not isinstance(df.index,pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    return df
def _safe_atr(h,l,c,ln):
    try:
        a = ta.atr(h,l,c,length=ln)
        if a is None or a.empty: return 0.0
        v = a.iloc[-1]; return float(v) if pd.notna(v) else 0.0
    except Exception as e: _log_exc('atr',e); return 0.0

# Guardiões
def guardiao_dow(df,ctx,modo):
    if modo!='black_belt': return True,None
    dow = detectar_estrutura_dow(df)
    return (ctx or (dow and dow['tendencia_dow']=='ALTA')),'Dow'
def guardiao_elliott(df,ctx,modo):
    if modo!='black_belt': return True,None,False
    sh,sl = [],[]
    for j in range(5,len(df)-5):
        if df['High'].values[j]>=max(df['High'].values[j-5:j+6]): sh.append(df['High'].values[j])
        if df['Low'].values[j]<=min(df['Low'].values[j-5:j+6]): sl.append(df['Low'].values[j])
    val,_ = validar_elliott(df,sl,sh)
    return (ctx or val),'Elliott',val
def guardiao_fibonacci(df,ent):
    fib = calcular_fibonacci_retracao(df)
    if fib: return (fib['61.8%']<=ent<=fib['38.2%']),'Fibonacci'
    return True,None
def guardiao_retangulo(df,ent,modo):
    if modo!='black_belt': return True,None
    ret = detectar_retangulo(df)
    if ret and 'suporte' in ret: return (ent<=ret['suporte']*1.05),'Retângulo'
    return True,None
def guardiao_estocastico(df,modo):
    if modo!='black_belt': return True,None
    k,_ = calcular_estocastico(df)
    return (k is not None and k<30),'Estocástico'
def guardiao_medias(df,ent,modo):
    if modo!='black_belt': return True,None
    mm = df['Close'].rolling(200).mean().iloc[-1]
    mma = df['Close'].rolling(200).mean().iloc[-5] if len(df)>=200 else mm
    return (pd.notna(mm) and ent>mm and mm>mma),'Médias'
def guardiao_gatilho(df):
    pc = analisar_candle(df.iloc[-1],df.iloc[-2] if len(df)>=2 else None)
    return (pc.get('martelo') or pc.get('engolfo_alta') or pc.get('kicker_alta') or pc.get('harami_alta')),'Gatilho',pc
def guardiao_payoff(ent,alv,stp,cst,dir='COMPRA',min=3.0):
    p = calcular_payoff_real(ent,alv,stp,cst,dir)
    return (p>=min),'Payoff',p
def guardiao_corda(df,ent,modo,dm=30.0):
    if modo!='black_belt': return True,None
    mm = df['Close'].rolling(200).mean().iloc[-1]
    if pd.notna(mm) and mm>0: return ((ent-mm)/mm*100<=dm),'Corda'
    return True,None
def guardiao_macd(df,usar):
    if not usar: return True,None
    try:
        m = ta.macd(df['Close'])
        if m is None or m.empty: return True,None
        return (m.iloc[:,0].iloc[-1]>m.iloc[:,1].iloc[-1]),'MACD'
    except Exception as e: _log_exc('macd',e); return True,None
def guardiao_setor(t,ct,mx):
    s = obter_setor(t); ok = ct.get(s,0)<mx
    return ok,f'Setor ({s})',s
def calcular_score_qualidade(r,dow,ell,fib,ret,sto,ma,zona,gat,pay,cor,ac,bb,ctx,arm,ef,reg):
    sc = 50
    if ctx: sc+=20
    if arm: sc+=10
    if ef and ef>0.8: sc+=15
    elif ef and ef>0.6: sc+=8
    if ell: sc+=10
    if dow and not ctx: sc+=5
    if reg=='acumulacao': sc+=5
    elif reg=='markup': sc+=3
    if pay and r.get('Payoff Real',0)>4.0: sc+=5
    return min(100,max(0,sc))

print("✅ Célula 3 carregada")

In [ ]:
# ==================== CÉLULA 4: EXECUÇÃO PRINCIPAL ====================
def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE) as f: cache = json.load(f)
            if (datetime.now()-datetime.fromisoformat(cache['timestamp'])).total_seconds()/3600<24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)")
                return cache['tickers']
        except: pass
    try:
        resp = requests.get("https://www.dadosdemercado.com.br/acoes",timeout=10,headers={'User-Agent':'Mozilla/5.0'})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content,'html.parser')
        tickers = [cells[0].text.strip().replace('.SA','') for row in soup.select('table tbody tr') if (cells:=row.find_all('td')) and not cells[0].text.strip().startswith('#') and cells[0].text.strip()]
        if tickers:
            with open(CACHE_TICKERS_FILE,'w') as f: json.dump({'timestamp':datetime.now().isoformat(),'tickers':tickers},f)
            logger.log(f"🌐 Scraping ({len(tickers)} ativos)")
            return tickers
    except Exception as e: logger.log(f"⚠️ Scraping: {str(e)[:80]}","WARN")
    return FALLBACK_TICKERS.copy()

def extrair_dataframe_ticker(data_raw,ticker):
    try:
        if isinstance(data_raw.columns,pd.MultiIndex):
            if ticker not in data_raw.columns.get_level_values(1) and ticker not in data_raw.columns.get_level_values(0): return None
            df = data_raw[ticker].copy()
        else: df = data_raw.copy()
        if isinstance(df.columns,pd.MultiIndex): df.columns = ['_'.join(col).strip() for col in df.columns.values]
        df.columns = [c.lower() for c in df.columns]
        rm = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
        df.rename(columns={k:rm.get(k,k) for k in df.columns if k in rm},inplace=True)
        return df
    except Exception as e: _log_exc(f'extrair {ticker}',e); return None

def resample_tf(df,freq,md=4,mdm=10):
    if df is None or df.empty: return None
    df = df.copy()
    if not isinstance(df.index,pd.DatetimeIndex):
        try: df.index = pd.to_datetime(df.index)
        except: return None
    hoje = datetime.now()
    if freq.startswith('W') and (hoje.weekday()<4 or (hoje.weekday()==4 and hoje.hour<18)):
        us = df.index[df.index.dayofweek==4]
        if len(us)>0:
            df = df.loc[:us[-1]]
            if df.empty: return None
    agg = {'Open':'first','High':'max','Low':'min','Close':'last','Volume':'sum'}
    dfr = df.resample(freq,closed='right',label='right').agg(agg)
    cnt = df.resample(freq,closed='right',label='right').count()['Close']
    dfr = dfr[cnt>=(md if freq.startswith('W') else mdm)]
    dfr = dfr.replace([np.inf,-np.inf],np.nan).dropna()
    return dfr[dfr['Close']>0]

# ---- FLUXO ----
logger.inicio("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
tickers_b3 = [t.replace('.SA','') for t in tickers_b3]
logger.fim("Coleta de Tickers",{'total':len(tickers_b3)})

tickers_yahoo = [t+".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.inicio("Filtro de Liquidez")
for i in range(0,len(tickers_yahoo),BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        dr = yf.download(batch,period='3mo',interval='1d',group_by='ticker',progress=False,auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS: continue
            df = extrair_dataframe_ticker(dr,t)
            if df is None or df.empty or 'Volume' not in df.columns: continue
            try:
                vm = df['Volume'].rolling(21).mean().iloc[-1]
                pc = df['Close'].iloc[-1]
                if pd.isna(vm) or pd.isna(pc) or pc<=0: continue
                if vm>=VOLUME_MINIMO_ACAO and (vm*pc)>=VOLUME_FINANCEIRO_MINIMO: tickers_liquidos.append(t)
            except Exception as e: _log_exc(f'liq {t}',e)
    except Exception as e: logger.log(f"Erro lote {i//BATCH}: {str(e)[:100]}","ERRO")
    time.sleep(1)
if len(tickers_liquidos)<10:
    logger.log("Fallback tickers","WARN")
    tickers_liquidos = [t+".SA" for t in FALLBACK_TICKERS[:20]]
logger.fim("Filtro de Liquidez",{'liquidos':len(tickers_liquidos)})

logger.inicio("Download Histórico")
data_d = {}
for i in range(0,len(tickers_liquidos),BATCH):
    batch = tickers_liquidos[i:i+BATCH]
    try:
        dr = yf.download(batch,period='5y',interval='1d',group_by='ticker',progress=False,auto_adjust=True)
        for t in batch:
            df = extrair_dataframe_ticker(dr,t)
            if df is not None and not df.empty: data_d[t] = df
    except Exception as e: _log_exc(f'download',e)
    time.sleep(1)
logger.fim("Download Histórico",{'sucesso':len(data_d)})

logger.inicio("Resample")
data_w,data_m = {},{}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            dfd = data_d[t].copy()
            data_w[t] = resample_tf(dfd,'W-FRI')
            if data_w[t] is not None: data_m[t] = resample_tf(dfd,'ME',mdm=10)
    except Exception as e: _log_exc(f'resample {t}',e)
logger.fim("Resample",{'semanais':len(data_w)})

nh_nl = 50.0
if tickers_liquidos:
    try:
        cnt = sum(1 for t in tickers_liquidos if t in data_w and data_w[t] is not None and not data_w[t].empty and len(data_w[t])>=50 and float(data_w[t]['Close'].iloc[-1])>float(data_w[t]['Close'].rolling(50).mean().iloc[-1]))
        nh_nl = round(cnt/len(tickers_liquidos)*100,1)
    except: pass
logger.log(f"📊 NH‑NL: {nh_nl}%")

logger.inicio("Regime")
regime_vol = 'BAIXA'
try:
    for sim in ["^BVSP","^IBOV","BOVA11.SA"]:
        try:
            ib = yf.download(sim,period='3mo',interval='1d',progress=False,auto_adjust=True)
            if not ib.empty and 'Close' in ib.columns:
                ibov = ib['Close'].dropna()
                if len(ibov)>=60: break
        except: pass
    if ibov is not None and len(ibov)>=60: regime_vol = detectar_regime_volatilidade(ibov)
except Exception as e: logger.warn(f"Regime: {e}")
PARAMS_ATIVOS.clear()
PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol=='ALTA' else PARAMS_BAIXA_VOL)
logger.fim("Regime",{'regime':regime_vol})

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO,PAYOFF_ESTIMADO,PARAMS_ATIVOS['kelly_frac'])

oportunidades_swing,status_ativos = [],[]
contagem_setores = {}
pc_min = PARAMS_ATIVOS.get('preco_minimo',PRECO_MINIMO)
rx_max = PARAMS_ATIVOS.get('risco_percentual_maximo',RISCO_PERCENTUAL_MAXIMO)
analisados = 0

logger.inicio("Análise de Setups")
for i,ticker in enumerate(tickers_liquidos):
    if i%20==0: logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}","DEBUG")
    df_w = data_w.get(ticker)
    if df_w is None or df_w.empty:
        status_ativos.append({'Ticker':ticker,'Direcao':'N/A','Status':'❌ Recusado','Filtro':'Sem dados','Detalhe':'Sem dados semanais'})
        continue
    analisados += 1
    dfn = _normalizar_dataframe(df_w)
    dfn['Eficiencia'] = calcular_eficiencia_candle(dfn)
    dfn['Regime'] = detectar_regime(dfn)
    ult = dfn.iloc[-1]
    entrada = float(ult['Close'])
    direcao = 'COMPRA'
    if pd.isna(entrada) or entrada<=0 or entrada<pc_min:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Preço','Detalhe':f'R$ {entrada:.2f} < R$ {pc_min:.2f}'})
        continue
    rl = float(dfn['Low'].rolling(window=min(52,len(dfn))).min().iloc[-1])
    reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
    ef = round(float(ult['Eficiencia']),2) if not pd.isna(ult['Eficiencia']) else None
    if reg not in [1,2] or ef is None or ef<0.6:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Confluência','Detalhe':f'Regime {reg}, Eficiência {ef}'})
        continue
    mm = dfn['Close'].rolling(200).mean().iloc[-1]
    if pd.notna(mm) and entrada<mm:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'MM200','Detalhe':f'R$ {entrada:.2f} < R$ {mm:.2f}'})
        continue
    if PARAMS_ATIVOS.get('exigir_volume_anormal',False) and not detectar_volume_anormal(dfn):
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Volume','Detalhe':'Volume < 1.5x média'})
        continue
    res_lta = calcular_lta_adaptativo(dfn)
    if res_lta is None:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'LTA','Detalhe':'LTA não encontrada'})
        continue
    lta_val = res_lta[0]
    is_arm,_ = detectar_armadilha_lta(dfn,rl,BANDA_ZONA_PCT)
    ctx_trap = is_arm
    atr = _safe_atr(dfn['High'],dfn['Low'],dfn['Close'],14) or entrada*0.02
    stop_atr = entrada-1.8*atr
    sw = detectar_swing_low(dfn,12)
    sc = [s for s in [stop_atr,sw] if s is not None and s>0 and s<entrada]
    if not sc:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Stop','Detalhe':'Sem stop válido'})
        continue
    stop_f = max(sc)
    risco = entrada-stop_f
    if risco/entrada<RISCO_PERCENTUAL_MINIMO or risco/entrada>rx_max:
        status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':'Risco','Detalhe':f'Risco {risco/entrada*100:.1f}%'})
        continue
    alvo = entrada+risco*3
    # Guardiões
    ok_dow,lb_dow = guardiao_dow(dfn,ctx_trap,MODO_GEBRA)
    if not ok_dow: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_dow,'Detalhe':'Dow'}); continue
    ok_ell,lb_ell,ell_val = guardiao_elliott(dfn,ctx_trap,MODO_GEBRA)
    if not ok_ell: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_ell,'Detalhe':'Elliott'}); continue
    ok_fib,lb_fib = guardiao_fibonacci(dfn,entrada)
    if not ok_fib: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_fib,'Detalhe':'Fibonacci'}); continue
    ok_ret,lb_ret = guardiao_retangulo(dfn,entrada,MODO_GEBRA)
    if not ok_ret: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_ret,'Detalhe':'Retângulo'}); continue
    ok_sto,lb_sto = guardiao_estocastico(dfn,MODO_GEBRA)
    if not ok_sto: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_sto,'Detalhe':'Estocástico>30'}); continue
    ok_ma,lb_ma = guardiao_medias(dfn,entrada,MODO_GEBRA)
    if not ok_ma: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_ma,'Detalhe':'Médias'}); continue
    # WYCKOFF (NOVO)
    ok_w,lb_w,reg_w,info_w = guardiao_wyckoff(dfn,lta_val,BANDA_ZONA_PCT)
    if not ok_w:
        wx_det = info_w.get('evento','Fora da zona Wyckoff') if USAR_WYCKOFF_COMPLETO else 'Fora da zona Wyckoff'
        status_ativos.append({'Ticker':ticker,'Direcao':info_w.get('direcao',direcao) if USAR_WYCKOFF_COMPLETO else direcao,'Status':'❌ Recusado','Filtro':lb_w,'Detalhe':wx_det}); continue
    # Se Wyckoff completo, atualizar direção
    if USAR_WYCKOFF_COMPLETO: direcao = info_w.get('direcao','COMPRA')
    ok_gat,lb_gat,pad_c = guardiao_gatilho(dfn)
    if not ok_gat: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_gat,'Detalhe':'Gatilho'}); continue
    ok_pay,lb_pay,payoff = guardiao_payoff(entrada,alvo,stop_f,PARAMS_ATIVOS['custos_pct'],direcao)
    if not ok_pay: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_pay,'Detalhe':f'Payoff {payoff:.1f}<3.0'}); continue
    ok_cor,lb_cor = guardiao_corda(dfn,entrada,MODO_GEBRA,DIST_CORDA_MAX)
    if not ok_cor: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_cor,'Detalhe':f'Corda>{DIST_CORDA_MAX}%'}); continue
    ok_mac,lb_mac = guardiao_macd(dfn,USAR_GUARDIAO_MACD)
    if not ok_mac: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_mac,'Detalhe':'MACD'}); continue
    ok_set,lb_set,setor = guardiao_setor(ticker,contagem_setores,MAX_ATIVOS_POR_SETOR)
    if not ok_set: status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'❌ Recusado','Filtro':lb_set,'Detalhe':f'Setor {setor}'}); continue
    # APROVADO
    contagem_setores[setor] = contagem_setores.get(setor,0)+1
    score = calcular_score_qualidade({},True,ell_val,True,True,True,True,True,True,True,True,True,True,ctx_trap,is_arm,ef,reg_w)
    padroes = [k for k,v in pad_c.items() if v] if pad_c else []
    if direcao=='COMPRA':
        inst = f'Se preço atingir R$ {entrada:.2f}, COMPRAR com stop R$ {stop_f:.2f} e alvo R$ {alvo:.2f}. Payoff {payoff}:1.'
    else:
        inst = f'Se preço atingir R$ {entrada:.2f}, VENDER com stop R$ {stop_f:.2f} e alvo R$ {alvo:.2f}. Payoff {payoff}:1.'
    setup = {'Ticker':ticker,'Direcao':direcao,'Entrada':round(entrada,2),'Stop Loss':round(stop_f,2),'Alvo Recomendado':round(alvo,2),'Payoff Real':payoff,'Score Qualidade':score,'Padrões Detectados':', '.join(padroes) if padroes else 'Nenhum','Instrucao':inst}
    oportunidades_swing.append(setup)
    status_ativos.append({'Ticker':ticker,'Direcao':direcao,'Status':'✅ APROVADO','Filtro':'Nenhum','Detalhe':inst})

logger.fim("Análise de Setups",{'analisados':analisados,'aprovados':len(oportunidades_swing)})
if oportunidades_swing: oportunidades_swing = sorted(oportunidades_swing,key=lambda x:x.get('Score Qualidade',0),reverse=True)[:MAX_SETUPS_POR_DIA]
logger.log(f"🎯 Swing: {len(oportunidades_swing)} | Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol}","RESULTADO")

# Relatório
def gerar_relatorio(sa,op,rv,nn,lad,kp):
    ls = []; ls.append("="*80); ls.append("RELATÓRIO GEBRA v9.1"); ls.append(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    ls.append(f"Regime: {rv} | NH-NL: {nn}% | Kelly: {kp*100:.2f}% | Wyckoff: {'LIGADO' if USAR_WYCKOFF_COMPLETO else 'DESLIGADO'}")
    ls.append(f"Oportunidades: {len(op)}"); ls.append("="*80); ls.append("")
    ap = [s for s in sa if 'APROVADO' in str(s.get('Status',''))]
    rc = [s for s in sa if 'Recusado' in str(s.get('Status',''))]
    if ap:
        ls.append(f"✅ APROVADOS ({len(ap)}):"); ls.append("")
        for s in ap: ls.append(f"  {s['Ticker']} | {s.get('Direcao','N/A')}"); ls.append(f"  📌 {s.get('Detalhe','')}"); ls.append("")
    if rc:
        ls.append(f"❌ RECUSADOS ({len(rc)}):"); ls.append("")
        mt = Counter([s.get('Filtro','?') for s in rc])
        ls.append("  Resumo:")
        for m,q in mt.most_common(): ls.append(f"    • {m}: {q}")
        ls.append(""); ls.append("  Detalhe:"); ls.append("")
        for s in rc: ls.append(f"  {s['Ticker']} | {s.get('Direcao','N/A')} | {s.get('Filtro','?')}");
            if s.get('Detalhe'): ls.append(f"  📝 {s.get('Detalhe','')}"); ls.append("")
    ls.append("--- FIM ---")
    return "\n".join(ls)

rel = gerar_relatorio(status_ativos,oportunidades_swing,regime_vol,nh_nl,{},kelly_pct)
with open('relatorio_detalhado.txt','w') as f: f.write(rel)

def enviar_email(ass,c):
    if not EMAIL_REMETENTE or not SENHA_APP: logger.warn("E-mail não configurado"); return
    try:
        msg = MIMEMultipart(); msg['From']=EMAIL_REMETENTE; msg['To']=EMAIL_REMETENTE; msg['Subject']=ass
        msg.attach(MIMEText(c,'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com',465) as srv: srv.login(EMAIL_REMETENTE,SENHA_APP); srv.send_message(msg)
        logger.log("📧 E-mail enviado")
    except Exception as e: _log_exc('Email',e)
enviar_email(f"📊 GEBRA v9.1 - {len(oportunidades_swing)} Ops - {datetime.now().strftime('%d/%m %H:%M')}",rel)

# Telegram
def fmt_tg(op,rv,kp):
    if not op: return f"📊 GEBRA v9.1\nNenhuma oportunidade.\nRegime: {rv} | Kelly: {kp*100:.1f}%"
    m = f"🚀 GEBRA - {datetime.now().strftime('%d/%m %H:%M')}\nRegime: {rv} | Kelly: {kp*100:.1f}%\n\n"
    for i,o in enumerate(op[:MAX_SETUPS_POR_DIA],1):
        m += f"{i}. <b>{o['Ticker']}</b> | {o['Direcao']}\n"
        m += f"   Entrada: R$ {o['Entrada']:.2f} | Stop: R$ {o['Stop Loss']:.2f}\n"
        m += f"   Alvo: R$ {o.get('Alvo Recomendado',0):.2f} | Payoff: {o.get('Payoff Real',0):.2f}\n"
        m += f"   Score: {o.get('Score Qualidade',0)}\n   📌 {o.get('Instrucao','')}\n\n"
    return m
enviar_telegram(fmt_tg(oportunidades_swing,regime_vol,kelly_pct))

gc.collect()
logger.resumo()
print("\n✅ Concluído. Relatório: relatorio_detalhado.txt")
print(f"   Wyckoff Completo: {'LIGADO' if USAR_WYCKOFF_COMPLETO else 'DESLIGADO'} | Sensibilidade: {WYCKOFF_SENSIBILIDADE}")